# Modelo de Predicción de Adopciones

El objetivo es construir un modelo de Machine Learning que pueda predecir la probabilidad de que un animal sea adoptado, basándonos en sus características.

**Pasos a seguir:**
1.  **Cargar** los datos limpios.
2.  **Preparar** los datos para el modelo (Encoding).
3.  **Dividir** los datos en sets de entrenamiento y prueba.
4.  **Entrenar** un modelo de clasificación.
5.  **Evaluar** el rendimiento del modelo.

In [1]:
# importación de librerías
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# esto es para ignorar las advertencias y que se vean bien los resultados
import warnings

warnings.filterwarnings("ignore")

In [3]:
# se carga el dataset limpio
df = pd.read_csv("./dataset/datos_limpios_refugio.csv")

# solo para verificar que se haya cargado bien
df.head()

,fecha_hora,tipo_resultado,tipo_animal,raza,color,edad_en_dias,esterilizado,sexo
0,2014-07-22 16:04:00,Transfer,Cat,Domestic Shorthair Mix,Orange Tabby,14,No,Macho
1,2013-11-07 11:47:00,Transfer,Dog,Beagle Mix,White/Brown,365,Si,Hembra
2,2014-06-03 14:20:00,Adoption,Dog,Pit Bull,Blue/White,365,Si,Macho
3,2014-06-15 15:50:00,Transfer,Dog,Miniature Schnauzer Mix,White,3285,Si,Macho
4,2014-07-07 14:04:00,Euthanasia,Other,Bat Mix,Brown,150,Desconocido,Desconocido


## Preparación de Datos

### 1. Definir el Problema (Variable Objetivo 'y')

En esta caso la columna `tipo_resultado` tiene muchas categorías (Adoption, Transfer, Euthanasia, etc.). Para lograr simplificar el modelo, es más fácil convertir esto en un problema **binario** (Sí/No).

Entonces lo que se hará es crear una nueva columna `fue_adopcion`.
* Será **1** (Sí) si el `tipo_resultado` fue 'Adoption'.
* Será **0** (No) para cualquier otro resultado (Transfer, Euthanasia, etc.).

In [4]:
# entonces se crea la variable objetivo 'y'
# 1 si es 'Adoption', 0 si es cualquier otra cosa
df["fue_adopcion"] = df["tipo_resultado"].apply(lambda x: 1 if x == "Adoption" else 0)

# ahora para comprobar, ver cuántos 1s y 0s existen
print(df["fue_adopcion"].value_counts())

fue_adopcion
0    45132
1    33112
Name: count, dtype: int64


In [5]:
# ahora entonces se guarda esta columna como nuestra 'y'
y = df["fue_adopcion"]

### 2. Seleccionar las Características 'X'

Ahora hay que elegir qué columnas usaremos para *predecir* si un animal será adoptado.

Para esto entonces se optó por las siguientes variables o columnas: `tipo_animal`, `edad_en_dias`, `esterilizado` y `sexo`.

**Nota Importante:** Excluimos `raza` y `color`. 
¿Por qué? Estas columnas tienen demasiados de valores únicos, lo que puede hacer que el modelo sea muy complejo y lento.

In [6]:
# seleccionamos solo las columnas que serán nuestras carecterísticas 'X'=features
features = ["tipo_animal", "edad_en_dias", "esterilizado", "sexo"]
X = df[features]

In [7]:
# para verificar se muestran las primeras filas de nuestras features
X.head()

,tipo_animal,edad_en_dias,esterilizado,sexo
0,Cat,14,No,Macho
1,Dog,365,Si,Hembra
2,Dog,365,Si,Macho
3,Dog,3285,Si,Macho
4,Other,150,Desconocido,Desconocido


### 3. Codificación de Variables Categóricas

El modelo de IA no entiende texto (ej. "Dog", "Cat", "Si", "No"). Es por esta razón que entonces debemos convertir estas categorías en números.

Usaremos una técnica llamada **"One-Hot Encoding"** (usando `pd.get_dummies`).

Esto básicamente convierte una columna como `tipo_animal` en múltiples columnas (ej. `tipo_animal_Dog`, `tipo_animal_Cat`), que solo tendrán0 o 1, dependiendo de la variable categórica que se trate.

In [8]:
# se aplica One-Hot Encoding a X

# IMPORTANTE: 'edad_en_dias' es numérica, así que la función la ignorará y la mantendrá como está.
X_encoded = pd.get_dummies(X)

In [9]:
# para verificar entonces se muestran las primeras filas del resultado.
print("Características (features) codificadas:")
X_encoded.head()

Características (features) codificadas:


,edad_en_dias,tipo_animal_Bird,tipo_animal_Cat,tipo_animal_Dog,tipo_animal_Livestock,tipo_animal_Other,esterilizado_Desconocido,esterilizado_No,esterilizado_Si,sexo_Desconocido,sexo_Hembra,sexo_Macho
0,14,False,True,False,False,False,False,True,False,False,False,True
1,365,False,False,True,False,False,False,False,True,False,True,False
2,365,False,False,True,False,False,False,False,True,False,False,True
3,3285,False,False,True,False,False,False,False,True,False,False,True
4,150,False,False,False,False,True,True,False,False,True,False,False


## Entrenamiento del Modelo

### 1. Dividir los Datos (Train/Test Split)

Siempre recordar que no podemos probar nuestro modelo con los mismos datos que usamos para entrenarlo. Entonces lo que vamos hacer es dividir nuestros datos (X e y) en dos sets:
1.  **Set de Entrenamiento (80%):** Los datos que el modelo "verá" para aprender.
2.  **Set de Prueba (20%):** Los datos que "ocultamos". El modelo nunca los verá, y los usaremos al final para evaluar qué tan bueno es prediciendo.

In [10]:
# entonces se dividen los datos: 80% para entrenamiento, 20% para prueba

# random_state=42 es un número fijo para que, en caso se vuelva a ejecutar nuevamente, la división sea la misma.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

In [11]:
# para verificar el tamaño de los sets creados
print(f"Tamaño de X_train (entrenamiento): {X_train.shape}")
print(f"Tamaño de X_test (prueba): {X_test.shape}")

Tamaño de X_train (entrenamiento): (62595, 12)
Tamaño de X_test (prueba): (15649, 12)


### 2. Entrenar el Modelo (Regresión Logística)

Ahora para entrenar el modelo se hará de la siguiente manera:
1.  Creamos una instancia de nuestro modelo (`LogisticRegression`).
2.  Usamos el comando `.fit()` para entrenarlo, pasándole nuestros datos de **entrenamiento** (X_train, y_train).

El modelo ahora analizará estos datos y "aprenderá" la relación entre las características (features: edad, sexo, etc.) y el resultado (si fue adoptado o no).

In [12]:
# 1. se crea el modelo
modelo = LogisticRegression()

In [13]:
# 2. se pone a entrenar el modelo con los datos de entrenamiento
modelo.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Evaluación del Modelo

Ahora que el modelo está entrenado, entonces vamos a probarlo con los datos que "ocultamos" (el set de prueba: `X_test` y `y_test`).
1.  Usamos `.predict()` en `X_test` para obtener las predicciones del modelo.
2.  Comparamos esas predicciones con los resultados reales (`y_test`).

In [14]:
# 1. se hace que el modelo prediga los resultados para el set de prueba
y_predicciones = modelo.predict(X_test)

In [15]:
# 2. para verificar, entonces se observan las primeras 10 predicciones vs. lo que realmente pasó
print("--- Primeras 10 Predicciones ---")
print(f"Predicciones: {y_predicciones[:10]}")
print(f"Valores Reales: {y_test.values[:10]}")

--- Primeras 10 Predicciones ---
Predicciones: [1 1 1 0 1 0 0 0 0 0]
Valores Reales: [1 1 1 0 1 0 1 0 0 0]


In [16]:
# ya con las predicciones hechas, ahora se evalúa el rendimiento del modelo
accuracy = accuracy_score(y_test, y_predicciones)
print(f"\nPrecisión Total: {accuracy * 100:.2f}%")


Precisión Total: 74.34%


In [17]:
# ahora se muestra un reporte de clasificación detallado
print("Reporte de Clasificación:")
# target_names es para poner etiquetas, (0=No Adopción, 1=Adopción)
print(
    classification_report(
        y_test, y_predicciones, target_names=["No Adopción", "Adopción"]
    )
)

Reporte de Clasificación:
              precision    recall  f1-score   support

 No Adopción       0.85      0.68      0.75      9050
    Adopción       0.65      0.84      0.73      6599

    accuracy                           0.74     15649
   macro avg       0.75      0.76      0.74     15649
weighted avg       0.77      0.74      0.74     15649



## Nota
Cómo se pudo observar el modelo actual  precisión del 74.34%, esto porque en el entrenamiento y en la preparación de los datos se omitieron las características o features, color y raza. En este caso raza tiene demasiados valores únicos, pero con el objetivo de mejorar la precisión del modelo se volverá a entrenar pero ahora tomando la raza y el color, pero para raza solo las 10 más comunes, al igual con color.

In [18]:
# solo se vuelve a asegurar que el dataset se haya cargado bien
df.head()

,fecha_hora,tipo_resultado,tipo_animal,raza,color,edad_en_dias,esterilizado,sexo,fue_adopcion
0,2014-07-22 16:04:00,Transfer,Cat,Domestic Shorthair Mix,Orange Tabby,14,No,Macho,0
1,2013-11-07 11:47:00,Transfer,Dog,Beagle Mix,White/Brown,365,Si,Hembra,0
2,2014-06-03 14:20:00,Adoption,Dog,Pit Bull,Blue/White,365,Si,Macho,1
3,2014-06-15 15:50:00,Transfer,Dog,Miniature Schnauzer Mix,White,3285,Si,Macho,0
4,2014-07-07 14:04:00,Euthanasia,Other,Bat Mix,Brown,150,Desconocido,Desconocido,0


Ahora entonces se hará lo siguiente:

1. Crear la y (objetivo), esta se queda igual que antes.
2. Crear y preparar 'X' pero ahora incluyendo raza y color.

**Importante** se aplicará también la técnica de agrupación antes de hacer el encoding, esto para tomar únicamente las 10 más comunes.

In [19]:
# 'y' se queda igual que antes
df["fue_adopcion"] = df["tipo_resultado"].apply(lambda x: 1 if x == "Adoption" else 0)
y = df["fue_adopcion"]

In [20]:
# ahora para 'X' se incluyen raza y color, que es el cambio clave
features = ["tipo_animal", "edad_en_dias", "esterilizado", "sexo", "raza", "color"]
X = df[features].copy()  # se hace una copia para evitar que se modifique el df original

X.head()

,tipo_animal,edad_en_dias,esterilizado,sexo,raza,color
0,Cat,14,No,Macho,Domestic Shorthair Mix,Orange Tabby
1,Dog,365,Si,Hembra,Beagle Mix,White/Brown
2,Dog,365,Si,Macho,Pit Bull,Blue/White
3,Dog,3285,Si,Macho,Miniature Schnauzer Mix,White
4,Other,150,Desconocido,Desconocido,Bat Mix,Brown


In [21]:
# Bloque A (NUEVO): Ingeniería de Característica 'es_mix'


# La función revisa si la palabra 'Mix' o el símbolo '/' (ej. Poodle/Beagle)
# están en el texto de la raza.
def chequear_mix(texto_raza):
    texto_raza = str(texto_raza)  # Aseguramos que sea string
    if "Mix" in texto_raza or "/" in texto_raza:
        return 1  # Sí es mezcla
    else:
        return 0  # No es mezcla (raza pura)


# Creamos la nueva columna 'es_mix' en X
X["es_mix"] = X["raza"].apply(chequear_mix)

print("--- Se creó la nueva feature 'es_mix' ---")
print(X["es_mix"].value_counts())

--- Se creó la nueva feature 'es_mix' ---
es_mix
1    72997
0     5247
Name: count, dtype: int64


In [22]:
# Bloque B (NUEVO): Ingeniería de Característica 'categoria_edad'

# 1. Definimos los "cortes" (bins) en días:
# 0-6 meses (Cachorro/Gatito), 6m-3 años (Joven), 3-8 años (Adulto), 8+ (Senior)
bins = [0, 180, 1095, 2920, 99999]  # Los números están en días
labels = ["Cachorro", "Joven", "Adulto", "Senior"]

# 2. Creamos la nueva columna 'categoria_edad' usando pd.cut
X["categoria_edad"] = pd.cut(X["edad_en_dias"], bins=bins, labels=labels, right=False)

# 3. Llenamos cualquier valor 'NaN' que pudiera crearse (ej. edad 0)
X["categoria_edad"] = X["categoria_edad"].fillna("Cachorro")

# 4. ¡MUY IMPORTANTE! Eliminamos la columna numérica original
X = X.drop("edad_en_dias", axis=1)

print("\n--- Se creó 'categoria_edad' y se eliminó 'edad_en_dias' ---")
print(X["categoria_edad"].value_counts())


--- Se creó 'categoria_edad' y se eliminó 'edad_en_dias' ---
categoria_edad
Joven       31950
Cachorro    26472
Adulto      14185
Senior       5637
Name: count, dtype: int64


### Agrupación de valores relevantes
Anteriormente se dejó fuera raza y color, esto porque la columna raza tiene miles de valores únicos ("Pit Bull Mix", "Beagle", "Labrador Mix", entre otros). Entonces si se aplica `get_dummies` nuevamente, se estarían creando miles de columnas nuevas, esto porque sería una por cada raza y el modelo colapsaría.

**La Solución** realmente importan las razas más comunes. Entonces se van a agrupar y úncamente nos quedaremos con las 10 razas más comunes y las 10 combinaciones de colores más comunes. Y todas las demás se van a etiquetar como "Otra_Raza" u "Otro_Color".

In [23]:
# 1. se agrupa 'raza'
# se obtiene el 'index' (los nombres) de las 10 razas más comunes
top_10_razas = X["raza"].value_counts().nlargest(10).index

# y si la raza no está en esa lista, la reemplazamos con 'Otra_Raza'
X["raza"] = X["raza"].apply(lambda x: x if x in top_10_razas else "Otra_Raza")


print("--- Nuevas categorías de Raza ---")
print(X["raza"].value_counts())

--- Nuevas categorías de Raza ---
raza
Otra_Raza                    30656
Domestic Shorthair Mix       23332
Pit Bull Mix                  6133
Chihuahua Shorthair Mix       4733
Labrador Retriever Mix        4607
Domestic Medium Hair Mix      2323
German Shepherd Mix           1892
Bat Mix                       1283
Domestic Longhair Mix         1228
Australian Cattle Dog Mix     1059
Siamese Mix                    998
Name: count, dtype: int64


In [24]:
# 2. ahora se agrupa 'color', de la misma forma
# se obtiene el 'index' (los nombres) de los 10 colores más comunes
top_10_colores = X["color"].value_counts().nlargest(10).index

# de igual manera si el color no está en la lista, entonces se reemplaza con 'Otro_Color'
X["color"] = X["color"].apply(lambda x: x if x in top_10_colores else "Otro_Color")

print("\n--- Nuevas categorías de Color ---")
print(X["color"].value_counts())


--- Nuevas categorías de Color ---
color
Otro_Color           41325
Black/White           8151
Black                 6600
Brown Tabby           4445
Brown                 3483
White                 2784
Brown/White           2444
Tan/White             2394
Brown Tabby/White     2338
Orange Tabby          2180
White/Black           2100
Name: count, dtype: int64


In [25]:
# se aplica One-Hot Encoding a X nuevamente, ahora con raza y color incluidos
X_encoded = pd.get_dummies(X)

print("\nCaracterísticas (features) codificadas \n¡Ahora con raza y color!:")
X_encoded.head()


Características (features) codificadas 
¡Ahora con raza y color!:


,es_mix,tipo_animal_Bird,tipo_animal_Cat,tipo_animal_Dog,tipo_animal_Livestock,tipo_animal_Other,esterilizado_Desconocido,esterilizado_No,esterilizado_Si,sexo_Desconocido,...,color_Brown/White,color_Orange Tabby,color_Otro_Color,color_Tan/White,color_White,color_White/Black,categoria_edad_Cachorro,categoria_edad_Joven,categoria_edad_Adulto,categoria_edad_Senior
0,1,False,True,False,False,False,False,True,False,False,...,False,True,False,False,False,False,True,False,False,False
1,1,False,False,True,False,False,False,False,True,False,...,False,False,True,False,False,False,False,True,False,False
2,0,False,False,True,False,False,False,False,True,False,...,False,False,True,False,False,False,False,True,False,False
3,1,False,False,True,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
4,1,False,False,False,False,True,True,False,False,True,...,False,False,False,False,False,False,True,False,False,False


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=50
)

In [27]:
modelo = RandomForestClassifier(n_estimators=100, random_state=42)

In [28]:
modelo.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [29]:
y_predicciones = modelo.predict(X_test)

In [30]:
accuracy = accuracy_score(y_test, y_predicciones)
print(f"\nPrecisión Total: {accuracy * 100:.2f}%")


Precisión Total: 76.78%


In [31]:
# ahora se muestra un reporte de clasificación detallado
print("Reporte de Clasificación:")
# target_names es para poner etiquetas, (0=No Adopción, 1=Adopción)
print(
    classification_report(
        y_test, y_predicciones, target_names=["No Adopción", "Adopción"]
    )
)

Reporte de Clasificación:
              precision    recall  f1-score   support

 No Adopción       0.81      0.78      0.80      9010
    Adopción       0.72      0.75      0.73      6639

    accuracy                           0.77     15649
   macro avg       0.76      0.77      0.76     15649
weighted avg       0.77      0.77      0.77     15649



**Intento de Mejora**
<br>
Ahora que se agrego raza y color al modelo se tuvo un incremento en la precisión del modelo, sin embargo, tampoco fue un aumento tan significativo. Entonces necesitamos un modelo más avanzado que esté diseñado para manejar texto y categorías con muchos valores únicos nativamente. 

El modelo a implementar será LightGBM (`LGBMClassifier`). La ventaja con este modelo es que si le pasamos los datos como texto (ej. "Pit Bull Mix"), él solo, internamente, sabe cómo manejarlos. No es necesario realizar la agrupacipon de "Otra_Raza".

In [ ]:
# reinicio con nuevo modelo LightGBM
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier

# 1. ahora se carga el dataset limpio nuevamente
df_lgbm = pd.read_csv("./dataset/datos_limpios_refugio.csv")

# 2. se crea la variable objetivo 'y'
df_lgbm["fue_adopcion"] = df_lgbm["tipo_resultado"].apply(
    lambda x: 1 if x == "Adoption" else 0
)
y = df_lgbm["fue_adopcion"]

In [ ]:
# 3. se seleccionan las características 'X', incluyendo raza y color
features = ["tipo_animal", "raza", "color", "esterilizado", "sexo"]
X = df_lgbm[features].copy()


# 4. se usa "es_mix", esto porque ayuda al modelo a entender mejor las razas
def chequear_mix(texto_raza):
    texto_raza = str(texto_raza)
    # se usa 'Si'/'No' para que sea categórico
    if "Mix" in texto_raza or "/" in texto_raza:
        return "Si"
    else:
        return "No"


X["es_mix"] = X["raza"].apply(chequear_mix)


# 5. se cargara "edad_en_dias" de nuevo para crear "categoria_edad", esto porque se usará bins
X["edad_en_dias"] = pd.read_csv("./dataset/datos_limpios_refugio.csv")["edad_en_dias"]

bins = [0, 180, 1095, 2920, 99999]
labels = ["Cachorro", "Joven", "Adulto", "Senior"]
X["categoria_edad"] = pd.cut(X["edad_en_dias"], bins=bins, labels=labels, right=False)
X["categoria_edad"] = X["categoria_edad"].fillna("Cachorro")

# se eliminan ahora las columnas que ya no se necesitan
X = X.drop(["edad_en_dias"], axis=1)

X.head()

,tipo_animal,raza,color,esterilizado,sexo,es_mix,categoria_edad
0,Cat,Domestic Shorthair Mix,Orange Tabby,No,Macho,Si,Cachorro
1,Dog,Beagle Mix,White/Brown,Si,Hembra,Si,Joven
2,Dog,Pit Bull,Blue/White,Si,Macho,No,Joven
3,Dog,Miniature Schnauzer Mix,White,Si,Macho,Si,Senior
4,Other,Bat Mix,Brown,Desconocido,Desconocido,Si,Cachorro


In [ ]:
# 6. ahora todas las columnas de tipo 'object' se convierten a 'category' para LightGBM, ya que este modelo maneja muy bien este tipo de datos

for col in X.columns:
    X[col] = X[col].astype("category")

print("--- Tipos de datos de X (listos para LGBM) ---")
X.info()

--- Tipos de datos de X (listos para LGBM) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78244 entries, 0 to 78243
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   tipo_animal     78244 non-null  category
 1   raza            78244 non-null  category
 2   color           78244 non-null  category
 3   esterilizado    78244 non-null  category
 4   sexo            78244 non-null  category
 5   es_mix          78244 non-null  category
 6   categoria_edad  78244 non-null  category
dtypes: category(7)
memory usage: 790.0 KB


In [41]:
# 7. ahora se preparan los datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. ahora se crea el nuevo modelo LightGBM y se entrena
# 'balanced' usa los valores de `y` para ajustar automáticamente los pesos de forma
# inversamente proporcional a las frecuencias de clase en los datos de entrada
modelo_lgbm = LGBMClassifier(class_weight="balanced", random_state=42)

modelo_lgbm.fit(X_train, y_train)

# 9. una vez entrenado, se hacen las predicciones
y_predicciones = modelo_lgbm.predict(X_test)

# ya con las predicciones hechas, ahora se evalúa el rendimiento del modelo
accuracy = accuracy_score(y_test, y_predicciones)

print(f"\n--- RESULTADO CON LIGHTGBM ---")
print(f"Precisión Total: {accuracy * 100:.2f}%")
print("Reporte de Clasificación:")
print(
    classification_report(
        y_test, y_predicciones, target_names=["No Adopción", "Adopción"]
    )
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 26513, number of negative: 36082
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001141 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1074
[LightGBM] [Info] Number of data points in the train set: 62595, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000

--- RESULTADO CON LIGHTGBM ---
Precisión Total: 75.48%
Reporte de Clasificación:
              precision    recall  f1-score   support

 No Adopción       0.85      0.70      0.77      9050
    Adopción       0.